# Session 5 notebook: when characteristics overlap

**Question.** Combining two signals, value and momentum, by choosing the mix with the highest Sharpe ratio is easy. What happens when you try the same thing with 60 characteristics, many of them near-copies of each other?

| Part | What you do | When |
|---|---|---|
| 1 | How long does it take to tell a real premium from luck? | before class, 10 min |
| 2 | Measure how much the characteristics overlap | in class |
| 3 | Barra with 60 characteristics: OLS vs. ridge | in class |
| 4 | From 2 signals to 60: the optimal mix, with and without shrinkage (Kozak, Nagel and Santosh) | in class |
| 5 | Discuss | in class |

**Data** (`week3_data.zip`, link on the course notes page): the Session 3 panel (same stocks, returns and 8 Barra styles) plus 52 characteristics from the Open Source Asset Pricing data of Chen and Zimmermann (Reading #24), 2015-01 to 2026-01.

Cells marked **Your turn** work as written. Change the value, rerun, and say what changed and why.

## Part 0. Setup (before class)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

# The class data (week3_data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to week3_data.zip between the quotes, then run this cell

import re, zipfile, urllib.request
DATA_DIR = Path("data")
if not (DATA_DIR / "w3_panel.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK)
    assert fid, "Paste the week3_data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "week3_data.zip")
    assert zipfile.is_zipfile("week3_data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("week3_data.zip").extractall(".")          # creates ./data/
    print("downloaded and unzipped week3_data.zip")

panel = pd.read_parquet(DATA_DIR / "w3_panel.parquet")
ff = pd.read_parquet(DATA_DIR / "ff_factors_monthly.parquet").set_index("month")
signals = pd.read_csv(DATA_DIR / "w3_signals.csv")

STYLES = ["size", "value", "momentum", "beta", "resid_vol", "leverage", "liquidity", "profitability"]
CZ = signals["acronym"].tolist()
CHARS = STYLES + CZ
GROUP = {**{s: "barra" for s in STYLES}, **dict(zip(signals["acronym"], signals["group"]))}
print(f"panel: {len(panel):,} stock-months, {panel.month.nunique()} months, {panel.month.min()} to {panel.month.max()}, "
      f"about {len(panel) // panel.month.nunique():,} stocks a month, {len(CHARS)} characteristics")

In [ ]:
# Put every characteristic on the same scale: each month, mean 0 and standard deviation 1 across stocks.
# (The Barra styles were cap-weighted z-scores and the new ones are ranks; this makes them comparable.)
panel[CHARS] = panel.groupby("month")[CHARS].transform(lambda x: (x - x.mean()) / x.std()).astype("float32")
months = sorted(panel.month.unique())
signals[["acronym", "group", "description", "paper"]].head(12)

**Read the signals table.** The 52 new characteristics come in groups: several value ratios, several momentum windows, several volatility measures, several liquidity measures. Each was published as a separate anomaly. Chen and Zimmermann *sign* every signal so that higher values go with higher average returns in the original paper. So `DolVol` is high for stocks with *low* dollar volume, and `Illiquidity` is high for illiquid stocks: same idea, same sign. All 52 are listed in `signals`; the other 8 are Session 3's Barra styles.

**Where the data come from (Reading #24).** Chen and Zimmermann rebuild the published predictors from CRSP and Compustat: 319 characteristics from 153 papers. For the 161 that were clearly significant in the original paper, 98% of their rebuilt long-short portfolios have t > 1.96. Those are in-sample t-statistics, so keep Part 1 in mind. Across their whole library, most pairs of predictors are nearly uncorrelated. We picked our 52 in ten economic groups with close cousins *on purpose*, because this session is about what overlap does to a regression.

## Part 1. How long until a premium is "real"? (before class)

For a strategy with annualized Sharpe ratio SR, observed for T years, the t-statistic on its average return is roughly

> t &asymp; SR &times; &radic;T

So the number of years you need before t reaches 2 is (2 / SR)&sup2;. The cell below turns each characteristic into a long-short portfolio (each stock's weight is its standardized characteristic, divided by the number of stocks) and computes its Sharpe ratio and t-statistic over our 11 years.

In [ ]:
def char_portfolios(panel, chars):
    """Each month, one long-short portfolio per characteristic: weight = standardized characteristic / number of stocks."""
    return panel.groupby("month").apply(lambda g: pd.Series(g[chars].values.T @ g["y"].values / len(g), index=chars),
                                        include_groups=False)

F = char_portfolios(panel, CHARS)                       # months x 60 returns
sr = F.mean() / F.std() * np.sqrt(12)
tstat = F.mean() / F.std() * np.sqrt(len(F))
summary = pd.DataFrame({"group": [GROUP[c] for c in CHARS], "SR (annual)": sr, "t-stat": tstat,
                        "years for t = 2": (2 / sr.abs()) ** 2})
print(f"{len(F)} months. Characteristics with |t| > 2: {(tstat.abs() > 2).sum()} of {len(CHARS)}")
summary.sort_values("t-stat", ascending=False).head(10)

In [ ]:
yrs = np.linspace(1, 40, 200)
fig, ax = plt.subplots(figsize=(7, 3.2))
for s in [0.3, 0.5, 1.0]:
    ax.plot(yrs, s * np.sqrt(yrs), label=f"SR = {s}")
ax.axhline(2, color="grey", ls="--", lw=1); ax.set_xlabel("years of data"); ax.set_ylabel("expected t-statistic")
ax.set_title("How long until a true premium shows t = 2?"); ax.legend(); plt.tight_layout(); plt.show()

**Check yourself.** (1) A manager with an information ratio of 0.5 needs 16 years to reach t = 2. Suppose such a manager lags its benchmark for four years. Is that evidence against skill? (2) Most of our 60 characteristics have |t| below 2 over 11 years. Does that mean they don't work? (3) The ones with the largest t-statistics: were they chosen *before* you looked at this sample?

## Part 2. How much do the characteristics overlap? (in class)

Session 3's Barra model used 8 styles. Practitioners use many more, and the literature has hundreds. Many are close cousins. Average the cross-sectional correlation matrix over months and look at the most correlated pairs.

In [ ]:
R = sum(g[CHARS].corr().values for _, g in panel.groupby("month")) / len(months)
R = pd.DataFrame(R, index=CHARS, columns=CHARS)
upper = R.where(np.triu(np.ones(R.shape, bool), 1)).stack()
top = upper.reindex(upper.abs().sort_values(ascending=False).index).head(12)
print(f"pairs with |correlation| > 0.7: {(upper.abs() > 0.7).sum()},  > 0.5: {(upper.abs() > 0.5).sum()}  (of {len(upper)})")
top.rename("average correlation").to_frame()

In [ ]:
order = sorted(CHARS, key=lambda c: (list(dict.fromkeys(GROUP.values())).index(GROUP[c]), c))
fig, ax = plt.subplots(figsize=(8.5, 7.5))
im = ax.imshow(R.loc[order, order], cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=90, fontsize=6)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=6)
fig.colorbar(im, shrink=0.7); ax.set_title("Average cross-sectional correlation, grouped"); plt.tight_layout(); plt.show()

eig = np.linalg.eigvalsh(R.values)[::-1]
print(f"eigenvalues of the correlation matrix: largest {eig[0]:.1f}, smallest {eig[-1]:.3f}, ratio {eig[0] / eig[-1]:,.0f}")
print(f"the first 10 of {len(eig)} directions carry {eig[:10].sum() / eig.sum():.0%} of the variation in characteristics")

**Check yourself.** Why is the Barra `momentum` style correlated 0.9 with `Mom12m`? Why is `size` strongly *negatively* correlated with `Illiquidity`?

### A primer: two near-twins in one regression

Take just two characteristics, `DolVol` and `Illiquidity` (correlation about 0.97), and run the monthly regression of next month's return on both. Any such regression can be rewritten, exactly, as a regression on two new variables:

- **average** = (DolVol + Illiquidity) / 2: how illiquid the stock is overall
- **gap** = (DolVol &minus; Illiquidity) / 2: which of the twins is higher

Nothing is lost: b<sub>1</sub>&middot;DolVol + b<sub>2</sub>&middot;Illiq = (b<sub>1</sub> + b<sub>2</sub>)&middot;average + (b<sub>1</sub> &minus; b<sub>2</sub>)&middot;gap. Same fit, same R&sup2;.

Now recall Chapter 9: a slope's standard error is inversely proportional to the spread of its x. The plot shows the problem. Stocks spread out a lot along the average and hardly at all along the gap.

In [ ]:
g = panel[panel.month == "2023-08"]
x1, x2 = g["DolVol"].values, g["Illiquidity"].values
avg, gap = (x1 + x2) / 2, (x1 - x2) / 2
print(f"correlation {np.corrcoef(x1, x2)[0, 1]:.2f};  spread (sd) of the average {avg.std():.2f}, of the gap {gap.std():.2f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x1, x2, s=4, alpha=0.3, color="gray")
for v, color, name in [(np.array([1, 1]) * avg.std() * 1.6, "teal", "average"), (np.array([1, -1]) * gap.std() * 1.6, "darkorange", "gap")]:
    ax.annotate("", xy=v, xytext=-v, arrowprops=dict(arrowstyle="<->", color=color, lw=2.5))
    ax.text(*(v * 1.1), name, color=color, fontsize=11)
ax.set_xlabel("DolVol (signed: low volume = high)"); ax.set_ylabel("Illiquidity")
ax.set_aspect("equal"); ax.set_title("One month, two near-twins"); plt.tight_layout(); plt.show()

Run the regression every month, three ways, and see how much each slope jumps around from month to month. The slope on DolVol is (average slope + gap slope)/2 and the slope on Illiquidity is (average slope &minus; gap slope)/2, so the gap's noise lands on both twins with opposite signs.

In [ ]:
rows = []
for m, g in panel.groupby("month"):
    x1, x2, y = g["DolVol"].values, g["Illiquidity"].values, g["y"].values
    one = np.ones(len(g))
    alone = np.linalg.lstsq(np.column_stack([one, x1]), y, rcond=None)[0][1]
    both = np.linalg.lstsq(np.column_stack([one, x1, x2]), y, rcond=None)[0][1:]
    ag = np.linalg.lstsq(np.column_stack([one, (x1 + x2) / 2, (x1 - x2) / 2]), y, rcond=None)[0][1:]
    rows.append({"DolVol alone": alone, "DolVol, with Illiquidity": both[0], "Illiquidity, with DolVol": both[1],
                 "average slope": ag[0], "gap slope": ag[1]})
slopes = pd.DataFrame(rows)
print("how much each slope moves month to month (sd, % per month):")
print((100 * slopes.std()).map("{:.1f}".format).to_string())
print(f"\ncorrelation of the DolVol and Illiquidity slopes: {slopes['DolVol, with Illiquidity'].corr(slopes['Illiquidity, with DolVol']):.2f}")

**Check yourself.** (1) Why is the average slope about as steady as DolVol alone? (2) Ridge pulls every slope toward zero by the same rule, and the pull matters most where the data are weakest. Which of the two slopes will ridge shrink more?

### From 2 characteristics to 60

The same rewrite works for any number of characteristics: 60 new variables, each a weighted mix of the originals, none correlated with another. Software finds them (principal components). The math names: each mix is an **eigenvector**, and its variance across stocks is its **eigenvalue**. The 60 variances add up to 60.

The cell below finds the mix with the smallest variance (the worst "gap") and the ratio of the largest variance to the smallest, the **condition number**. A slope's noise scales with 1/spread, so the worst slope is about &radic;(condition number) times noisier than the best.

**Ridge** adds &lambda; to every variance. Big variances barely change; tiny ones grow a lot. That is Part 3. In Part 4, KNS apply the same idea to the covariance of factor *returns*.

In [ ]:
d, V = np.linalg.eigh(R.values)                  # ascending: column 0 is the smallest-variance mix
print(f"variances: largest {d[-1]:.1f}, smallest {d[0]:.3f}; condition number {d[-1] / d[0]:,.0f}; worst slope about {np.sqrt(d[-1] / d[0]):.0f}x noisier than the best")
print("\nthe smallest-variance mix, largest weights:")
print(pd.Series(V[:, 0], index=CHARS).sort_values(key=abs, ascending=False).head(6).map("{:+.2f}".format).to_string())
print()
for lam in [0, 0.05, 0.1, 0.5, 1]:
    print(f"ridge with lambda {lam:<4}: smallest variance {d[0] + lam:.3f}, condition number {(d[-1] + lam) / (d[0] + lam):5,.0f}")

**Check yourself.** Is the worst gap among all 60 the same pair as in the primer? What do you expect when all 60 go into one regression?

## Part 3. Barra with 60 characteristics: OLS vs. ridge (in class)

Same regression as Session 3, one month at a time, across all stocks, weighted by the square root of market cap:

> r<sub>i</sub> = industry<sub>i</sub> + &Sigma;<sub>k</sub> x<sub>ik</sub> f<sub>k</sub> + u<sub>i</sub>

(Industry dummies here instead of country + constrained industries: the style returns are the same.) **Ridge** adds a penalty &lambda; &times; &Sigma; f<sub>k</sub>&sup2; to the least-squares objective, on the style returns only:

> f&#770; = (X&prime;WX + &lambda;P)<sup>&minus;1</sup> X&prime;Wr &nbsp;&nbsp; (P is 1 on the style columns, 0 on the industries)

&lambda; = 0 is ordinary weighted least squares. Kent Daniel's discussion (slide 12) makes the key point: each f&#770;<sub>k</sub> is the return on a *portfolio*, with weights given by row k of (X&prime;WX + &lambda;P)<sup>&minus;1</sup>X&prime;W. Below we track both the factor returns and how much leverage those portfolios need.

In [ ]:
def barra_month(g, lams):
    """One month. For each lambda: style returns, and the gross leverage (sum of |weights|) of each style's portfolio."""
    w = np.sqrt(g["me_prev"].values); w = w / w.sum()
    D = pd.get_dummies(g["industry"]).values.astype(float)
    X = np.hstack([D, g[CHARS].values.astype(float)])
    XtW = X.T * w
    P = np.r_[np.zeros(D.shape[1]), np.ones(len(CHARS))]
    out = {}
    for lam in lams:
        A_inv = np.linalg.inv(XtW @ X + lam * np.diag(P))
        weights = (A_inv @ XtW)[-len(CHARS):]              # one row of stock weights per style
        out[lam] = (weights @ g["y"].values, np.abs(weights).sum(axis=1))
    return out

LAMS = [0, 0.01, 0.03, 0.1, 0.3]
res = {m: barra_month(g, LAMS) for m, g in panel.groupby("month")}
fac = {lam: pd.DataFrame([res[m][lam][0] for m in months], index=months, columns=CHARS) for lam in LAMS}
lev = {lam: pd.DataFrame([res[m][lam][1] for m in months], index=months, columns=CHARS) for lam in LAMS}
print("done:", len(months), "monthly regressions x", len(LAMS), "values of lambda")

In [ ]:
pair = ["DolVol", "Illiquidity"]          # Your turn: try ["AM", "Leverage"] or ["IdioVol3F", "RealizedVol"]
rows = []
for lam in LAMS:
    f = fac[lam]
    rows.append({"lambda": lam,
                 f"corr of {pair[0]} and {pair[1]} factor returns": f[pair[0]].corr(f[pair[1]]),
                 f"vol of {pair[0]} (%/mo)": 100 * f[pair[0]].std(),
                 f"vol of {pair[1]} (%/mo)": 100 * f[pair[1]].std(),
                 "median style vol (%/mo)": 100 * f.std().median(),
                 "max style leverage": lev[lam].mean().max()})
pd.DataFrame(rows).set_index("lambda").round(2)

**Read it.** Under OLS (&lambda; = 0) two near-duplicate characteristics get factor returns that move in *opposite* directions: the regression cannot tell them apart, so it pays one and charges the other. The portfolios behind them are leveraged long-short bets on the small difference between two nearly identical characteristics. Ridge shrinks those offsetting bets: the correlation moves toward zero and the volatility falls. For a risk model or a return attribution (Session 3's Micron exercise), that stability is the point.

In [ ]:
# Does ridge help *predict* returns? Forecast next month's returns with the average of past style returns.
def forecast_test(f, start=36):
    ic, spread = [], []
    for t in range(start, len(months)):
        g = panel[panel.month == months[t]]
        pred = g[CHARS].values @ f.iloc[:t].mean().values      # only months before t
        ic.append(np.corrcoef(pred, g["y"])[0, 1])
        q = pd.qcut(pred, 10, labels=False)
        spread.append(g["y"].values[q == 9].mean() - g["y"].values[q == 0].mean())
    ic, spread = np.array(ic), np.array(spread)
    return {"mean IC": ic.mean(), "t (IC)": ic.mean() / ic.std() * np.sqrt(len(ic)),
            "top-bottom decile SR": spread.mean() / spread.std() * np.sqrt(12)}

pd.DataFrame({lam: forecast_test(fac[lam]) for lam in LAMS}).T.rename_axis("lambda").round(3)

**Check yourself.** Ridge barely changes the forecasts, and none of them is strong. Why? Each month's regression uses about 2,750 stocks to estimate 60 style returns, so the *cross-sectional* noise is small. The hard part is the *time-series* average of the factor returns: 11 years of noisy monthly numbers, in 60 correlated directions. That is Part 1's problem again, and it is where shrinkage matters.

## Part 4. From 2 signals to 60 (in class)

Start with two signals, value and momentum, and choose the split between them with the highest Sharpe ratio. With factor portfolios F, the Sharpe-maximizing weights are

> b = &Sigma;<sup>&minus;1</sup> &mu; &nbsp;&nbsp; (&mu; = average factor returns, &Sigma; = their covariance matrix)

These weights also define the stochastic discount factor (SDF). Start with the two signals.

In [ ]:
mkt = ff.loc[months, "mktrf"].values
Fx = F.copy()
for c in CHARS:                                         # remove each portfolio's market exposure, as in KNS
    Fx[c] = F[c] - np.polyfit(mkt, F[c].values, 1)[0] * mkt

def sharpe(r): return r.mean() / r.std() * np.sqrt(12)

two = Fx[["value", "momentum"]]
b2 = np.linalg.solve(two.cov().values, two.mean().values)
print("correlation of value and momentum portfolios:", round(two.corr().iloc[0, 1], 2))
print("Sharpe ratios  value:", round(sharpe(two["value"]), 2), " momentum:", round(sharpe(two["momentum"]), 2),
      " optimal mix:", round(sharpe(two.values @ b2), 2))
print("optimal weights (scaled to sum to 1):", (b2 / b2.sum()).round(2))

With two signals, &Sigma;<sup>&minus;1</sup>&mu; is harmless. Now use all 60. Estimate the weights on **2015-2020** and test them on **2021-2026**, data the estimate never saw (Session 4's protocol).

In [ ]:
train = Fx.index < "2021-01"
mu, S = Fx[train].mean().values, Fx[train].cov().values
mu_te, S_te = Fx[~train].mean().values, Fx[~train].cov().values

def evaluate(b):
    """In- and out-of-sample Sharpe ratio of the portfolio with weights b, and out-of-sample R-squared of
    the model's predicted average returns for the 60 portfolios (KNS's measure)."""
    r2 = 1 - ((mu_te - S_te @ b) ** 2).sum() / (mu_te ** 2).sum()
    return {"SR in-sample": sharpe(Fx[train].values @ b), "SR out-of-sample": sharpe(Fx[~train].values @ b), "OOS R2": r2}

b_ols = np.linalg.solve(S, mu)
print(f"train {train.sum()} months, test {(~train).sum()} months")
pd.Series(evaluate(b_ols)).round(2)

**Read it.** In-sample, the 60-signal mix has a Sharpe ratio no real strategy has ever had. Out of sample it collapses, and the negative R&sup2; says the estimated model predicts the 60 portfolios' average returns worse than predicting zero. This is Session 4's data mining, produced by one line of linear algebra.

**Kozak, Nagel and Santosh (2020)** put a prior on the SDF that rules out near-arbitrage: a portfolio with tiny risk should not earn a large premium. The posterior mean is a ridge estimator (their eq. 22):

> b&#770; = (&Sigma; + &gamma;I)<sup>&minus;1</sup> &mu;

In [ ]:
tau = np.trace(S)
GAMMAS = [0, 1e-4, 1e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0]             # gamma, as a fraction of trace(Sigma)
table = pd.DataFrame({g: evaluate(np.linalg.solve(S + g * tau * np.eye(len(CHARS)), mu)) for g in GAMMAS}).T
table.index.name = "gamma / trace"
table.round(2)

**Choose &gamma; without peeking.** The table above uses the test period to show what happens; you may not use it to *pick* &gamma;. Pick it by cross-validation inside 2015-2020 only: split the training years into five blocks, fit on four, score on the fifth, rotate.

In [ ]:
def cv_r2(g, folds=5):
    idx = np.array_split(np.arange(train.sum()), folds)
    Ft = Fx[train].values
    scores = []
    for k in range(folds):
        fit = np.setdiff1d(np.arange(train.sum()), idx[k])
        m1, S1 = Ft[fit].mean(0), np.cov(Ft[fit].T)
        m2, S2 = Ft[idx[k]].mean(0), np.cov(Ft[idx[k]].T)
        b = np.linalg.solve(S1 + g * np.trace(S1) * np.eye(len(CHARS)), m1)
        scores.append(1 - ((m2 - S2 @ b) ** 2).sum() / (m2 ** 2).sum())
    return np.mean(scores)

grid = np.logspace(-4, 2, 31)
cv = pd.Series([cv_r2(g) for g in grid], index=grid)
g_star = cv.idxmax()
print(f"gamma chosen by cross-validation on 2015-2020: {g_star:.3g} x trace(Sigma)")
pd.Series(evaluate(np.linalg.solve(S + g_star * tau * np.eye(len(CHARS)), mu))).round(2)

### 4.2 Why the prior is economics, not just statistics

Rotate the 60 portfolios into their principal components (PCs). PC *j* has variance d<sub>j</sub> (the eigenvalue). Two facts: (1) the in-sample squared Sharpe ratio that the unshrunk weights claim from PC *j* is &mu;<sub>j</sub>&sup2; / d<sub>j</sub>; (2) ridge multiplies each PC's weight by d<sub>j</sub> / (d<sub>j</sub> + &gamma;), so the low-variance PCs are shrunk the most.

In [ ]:
d, Q = np.linalg.eigh(S); d, Q = d[::-1], Q[:, ::-1]
mu_pc = Q.T @ mu
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].bar(range(1, len(d) + 1), mu_pc ** 2 / d * 12, color="#B24E17")
ax[0].set_title("Claimed squared Sharpe ratio by PC (2015-2020)"); ax[0].set_xlabel("PC (1 = most variance)")
ax[1].semilogy(range(1, len(d) + 1), d / (d + g_star * tau), marker=".", color="#0E6B63")
ax[1].set_title("Ridge keeps this fraction of each PC's weight (log scale)"); ax[1].set_xlabel("PC (1 = most variance)")
plt.tight_layout(); plt.show()
claimed = mu_pc ** 2 / d
print(f"share of the claimed in-sample squared Sharpe ratio from the last 20 PCs: {claimed[-20:].sum() / claimed.sum():.0%}")
P = Fx.values @ Q                                       # PC portfolio returns, all months
for name, sl in [("first 5 PCs", slice(0, 5)), ("last 20 PCs", slice(len(d) - 20, len(d)))]:
    r = P[:, sl] @ (mu_pc[sl] / d[sl])                  # unshrunk optimal mix of just these PCs
    print(f"{name:12s} Sharpe in-sample {sharpe(r[train]):5.2f}   out-of-sample {sharpe(r[~train]):5.2f}")

**Read it.** The unshrunk weights get most of their in-sample Sharpe ratio from the *last* PCs: combinations of near-duplicate characteristics with almost no variance. Out of sample, that claimed Sharpe ratio is mostly gone. A large premium on a nearly riskless portfolio would be a near-arbitrage, and arbitrageurs would trade it away, whether prices are set by rational investors or behavioral ones (KNS, Section 2.3). So the prior says: believe premia on the big, high-variance PCs; distrust them on the tiny ones. That is the same reason Kent Daniel gives for why quant shops shrink expected returns (Black-Litterman) and use factor risk models (slide 17).

## Part 5. Discuss (in class)

1. With two signals the covariance matrix is 2x2. At how many signals does &Sigma;<sup>&minus;1</sup>&mu; start to fail, and what decides it: the number of signals, the number of months, or how correlated the signals are?
2. In Part 3 ridge stabilized the Barra factor returns but barely changed the forecasts. In Part 4 it rescued them. What is different about the two problems?
3. The cross-validated &gamma; was chosen without looking at 2021-2026. Is that enough to trust the out-of-sample Sharpe ratio? Use Part 1: how many years would you need before a Sharpe ratio of that size has t = 2?
4. The portfolios here are equal-weighted and include small stocks, with no trading costs. Which way do those choices push the Sharpe ratios? (Session 8.)
5. KNS find that a few high-variance PCs price the cross-section well, but a few *characteristics* do not. Wednesday: Kelly, Pruitt and Su build a model on that idea, and Kozak and Nagel ask what it really shows.

**Your turn.** Change the train/test split to 2022 and rerun Part 4. Does the conclusion survive? Then try 2019: with 48 training months and 60 signals, &Sigma; cannot even be inverted. What does ridge do then?